# Domain-Adversarial NN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import lightning as L
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, f1_score, precision_recall_curve
from sklearn.preprocessing import RobustScaler
import os
import matplotlib.pyplot as plt
import seaborn as sns

# 1. setup and data loading

In [ ]:
print("="*60)
print("DOMAIN-ADVERSARIAL NEURAL NETWORK (DANN) - IMPROVED")
print("CROSS-DATASET AUTISM CLASSIFICATION")
print("="*60)

torch.manual_seed(42)
np.random.seed(42)

# Load balanced datasets
print("\nLoading balanced datasets...")
c4_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_matched_balanced.csv')
ybt_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_balanced_standardized.csv')

print(f"C4 balanced shape: {c4_balanced.shape}")
print(f"YBT balanced shape: {ybt_balanced.shape}")

# 2. robust feature engineering and alignment

In [ ]:
# --- Robust Feature Engineering for Maximum Overlap ---

def create_aggregate_features(df, prefix, n_items):
    item_cols = [f"{prefix}_{i}" for i in range(1, n_items+1) if f"{prefix}_{i}" in df.columns]
    if item_cols:
        df[f"{prefix}_total"] = df[item_cols].sum(axis=1)
    return df

for prefix, n_items in [('eq', 10), ('aq', 10), ('sqr', 10), ('spq', 10)]:
    c4_balanced = create_aggregate_features(c4_balanced, prefix, n_items)
    ybt_balanced = create_aggregate_features(ybt_balanced, prefix, n_items)

# D-score
for df in [c4_balanced, ybt_balanced]:
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['d_score'] = df['eq_total'] - df['sqr_total']

# Age-EQ interaction
for df in [c4_balanced, ybt_balanced]:
    if 'age' in df.columns and 'eq_total' in df.columns:
        df['age_x_eq'] = df['age'] * df['eq_total']

# Age-AQ interaction
for df in [c4_balanced, ybt_balanced]:
    if 'age' in df.columns and 'aq_total' in df.columns:
        df['age_x_aq'] = df['age'] * df['aq_total']

# AQ-EQ interaction
for df in [c4_balanced, ybt_balanced]:
    if 'aq_total' in df.columns and 'eq_total' in df.columns:
        df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']

# EQ/SQR ratio
for df in [c4_balanced, ybt_balanced]:
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)

# Log-transformed AQ total
for df in [c4_balanced, ybt_balanced]:
    if 'aq_total' in df.columns:
        df['log_aq_total'] = np.log1p(np.clip(df['aq_total'], a_min=0, a_max=None))

# Square root of age
for df in [c4_balanced, ybt_balanced]:
    if 'age' in df.columns:
        df['sqrt_age'] = np.sqrt(np.clip(df['age'], a_min=0, a_max=None))

# High AQ flag (e.g., AQ > 32)
for df in [c4_balanced, ybt_balanced]:
    if 'aq_total' in df.columns:
        df['high_aq'] = (df['aq_total'] > 32).astype(int)

# Now recompute feature lists
exclude_cols = ['autism_target']
c4_features = [col for col in c4_balanced.columns if col not in exclude_cols]
ybt_features = [col for col in ybt_balanced.columns if col not in exclude_cols]

common_features = sorted(list(set(c4_features) & set(ybt_features)))
print(f"Common features after robust alignment: {len(common_features)}")
print("Sample common features:", common_features[:10])

# Print missing features for debugging
missing_in_ybt = set(c4_features) - set(ybt_features)
missing_in_c4 = set(ybt_features) - set(c4_features)
print(f"Features in C4 but missing in YBT: {missing_in_ybt}")
print(f"Features in YBT but missing in C4: {missing_in_c4}")

# 3. data preperation 

In [ ]:
# === RECREATE DATA ARRAYS WITH UPDATED FEATURES ===

X_c4 = c4_balanced[common_features].values
y_c4 = c4_balanced['autism_target'].values
X_ybt = ybt_balanced[common_features].values
y_ybt = ybt_balanced['autism_target'].values

scaler = RobustScaler()
X_c4_scaled = scaler.fit_transform(X_c4)
X_ybt_scaled = scaler.transform(X_ybt)

X_train, X_val, y_train, y_val = train_test_split(
    X_c4_scaled, y_c4, test_size=0.2, stratify=y_c4, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_ybt_scaled shape: {X_ybt_scaled.shape}")
print("NaNs in X_train:", np.isnan(X_train).sum())
print("Infs in X_train:", np.isinf(X_train).sum())

# 4. gradient reversal layer

In [ ]:
print("\n" + "="*60)
print("IMPROVED GRADIENT REVERSAL LAYER")
print("="*60)

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

# 5. DA classifier 

In [ ]:
print("\n" + "="*60)
print("IMPROVED DOMAIN-ADVERSARIAL CLASSIFIER")
print("="*60)

class ImprovedDomainAdversarialClassifier(L.LightningModule):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout_rate=0.3, 
                 learning_rate=0.0005, alpha=1.0, domain_weight=0.05):
        super().__init__()
        self.save_hyperparameters()
        
        # Feature extractor
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.BatchNorm1d(hidden_dim)
            ])
            prev_dim = hidden_dim
        self.feature_extractor = nn.Sequential(*layers)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dims[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # Domain discriminator
        self.domain_discriminator = nn.Sequential(
            GradientReversalLayer(alpha),
            nn.Linear(hidden_dims[-1], 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        self.learning_rate = learning_rate
        self.alpha = alpha
        self.domain_weight = domain_weight
        self.class_criterion = nn.BCELoss()
        self.domain_criterion = nn.BCELoss()
        
    def forward(self, x):
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        domain_output = self.domain_discriminator(features)
        return class_output, domain_output
    
    def training_step(self, batch, batch_idx):
        x, y, domain = batch
        class_output, domain_output = self(x)
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        domain_loss = self.domain_criterion(domain_output.squeeze(), domain.float())
        total_loss = class_loss + self.domain_weight * domain_loss
        self.log('train_class_loss', class_loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_domain_loss', domain_loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_total_loss', total_loss, on_step=True, on_epoch=True, prog_bar=True)
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        self.log('train_f1', f1, on_epoch=True, prog_bar=True)
        return total_loss
    
    def validation_step(self, batch, batch_idx):
        x, y, domain = batch
        class_output, domain_output = self(x)
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        domain_loss = self.domain_criterion(domain_output.squeeze(), domain.float())
        total_loss = class_loss + self.domain_weight * domain_loss
        self.log('val_class_loss', class_loss, on_epoch=True, prog_bar=True)
        self.log('val_domain_loss', domain_loss, on_epoch=True, prog_bar=True)
        self.log('val_total_loss', total_loss, on_epoch=True, prog_bar=True)
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        try:
            auc = roc_auc_score(y.cpu(), class_output.squeeze().detach().cpu())
        except ValueError:
            auc = 0.5
        self.log('val_f1', f1, on_epoch=True, prog_bar=True)
        self.log('val_auc', auc, on_epoch=True, prog_bar=True)
        return total_loss
    
    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.7, patience=3
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_total_loss",
            },
        }

 # 6. domain aware data module

In [ ]:
print("\n" + "="*60)
print("IMPROVED DOMAIN-AWARE DATA MODULE")
print("="*60)

class ImprovedDomainAdaptationDataModule(L.LightningDataModule):
    def __init__(self, X_train, X_val, y_train, y_val, X_target, y_target, batch_size=64):
        super().__init__()
        self.X_train = X_train
        self.X_val = X_val
        self.y_train = y_train
        self.y_val = y_val
        self.X_target = X_target
        self.y_target = y_target
        self.batch_size = batch_size
    
    def setup(self, stage=None):
        self.train_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_train),
            torch.LongTensor(self.y_train),
            torch.zeros(len(self.X_train))
        )
        self.val_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_val),
            torch.LongTensor(self.y_val),
            torch.zeros(len(self.X_val))
        )
        self.target_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_target),
            torch.LongTensor(self.y_target),
            torch.ones(len(self.X_target))
        )
    
    def train_dataloader(self):
        combined_dataset = torch.utils.data.ConcatDataset([
            self.train_dataset, self.target_dataset
        ])
        return torch.utils.data.DataLoader(
            combined_dataset, 
            batch_size=self.batch_size, 
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )
    
    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_dataset, 
            batch_size=self.batch_size, 
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

# 7. model training

In [ ]:
print("\n" + "="*60)
print("IMPROVED DANN MODEL TRAINING")
print("="*60)

input_dim = len(common_features)
print("Model input_dim:", input_dim)  # Should print 44

model = ImprovedDomainAdversarialClassifier(
    input_dim=input_dim,
    hidden_dims=[128, 64, 32],
    dropout_rate=0.3,
    learning_rate=0.0005,
    alpha=1.0,
    domain_weight=0.05
)

data_module = ImprovedDomainAdaptationDataModule(
    X_train, X_val, y_train, y_val, X_ybt_scaled, y_ybt, batch_size=64
)

trainer = L.Trainer(
    max_epochs=100,
    accelerator='auto',
    devices=1,
    callbacks=[
        L.pytorch.callbacks.EarlyStopping(
            monitor='val_f1',
            patience=15,
            mode='max',
            verbose=True
        ),
        L.pytorch.callbacks.ModelCheckpoint(
            monitor='val_f1',
            mode='max',
            save_top_k=3,
            filename='best_dann_improved_{epoch:02d}_{val_f1:.3f}',
            verbose=True
        ),
        L.pytorch.callbacks.LearningRateMonitor(logging_interval='epoch')
    ],
    log_every_n_steps=25,
    enable_progress_bar=True,
    enable_model_summary=True,
    deterministic=True
)

print("Starting improved DANN training...")
trainer.fit(model, data_module)
print("Improved DANN training completed!")

# 8. model eval

In [ ]:
print("\n" + "="*60)
print("ENHANCED MODEL EVALUATION")
print("="*60)

best_model_path = trainer.checkpoint_callback.best_model_path
print(f"Loading best model from: {best_model_path}")

model = ImprovedDomainAdversarialClassifier.load_from_checkpoint(best_model_path)
model.eval()

val_predictions = []
val_probs = []
val_targets = []

with torch.no_grad():
    for batch in data_module.val_dataloader():
        x, y, domain = batch
        device = next(model.parameters()).device
        x = x.to(device)
        class_output, domain_output = model(x)
        val_probs.extend(class_output.squeeze().cpu().numpy())
        val_predictions.extend((class_output.squeeze() > 0.5).cpu().numpy())
        val_targets.extend(y.cpu().numpy())

val_probs = np.array(val_probs)
val_predictions = np.array(val_predictions)
val_targets = np.array(val_targets)

print("\nValidation Set Performance:")
print(classification_report(val_targets, val_predictions, zero_division=0))
print(f"ROC-AUC: {roc_auc_score(val_targets, val_probs):.3f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(val_probs, bins=50, alpha=0.7)
plt.title('Validation Prediction Probabilities Distribution')
plt.xlabel('Prediction Probability')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
plt.hist(val_probs[val_targets == 0], bins=30, alpha=0.7, label='Class 0', density=True)
plt.hist(val_probs[val_targets == 1], bins=30, alpha=0.7, label='Class 1', density=True)
plt.title('Prediction Probabilities by Class')
plt.xlabel('Prediction Probability')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

# 9. cross dataset testing

In [ ]:
print("\n" + "="*60)
print("CROSS-DATASET TESTING (C4 → YBT) - IMPROVED")
print("="*60)

X_ybt_tensor = torch.FloatTensor(X_ybt_scaled)
ybt_dataset = torch.utils.data.TensorDataset(
    X_ybt_tensor, torch.LongTensor(y_ybt), torch.ones(len(X_ybt_scaled))
)
ybt_dataloader = torch.utils.data.DataLoader(ybt_dataset, batch_size=64, shuffle=False)

ybt_predictions = []
ybt_probs = []
ybt_targets = []

model.eval()
device = next(model.parameters()).device

with torch.no_grad():
    for batch in ybt_dataloader:
        x, y, domain = batch
        x = x.to(device)
        class_output, domain_output = model(x)
        ybt_probs.extend(class_output.squeeze().cpu().numpy())
        ybt_predictions.extend((class_output.squeeze() > 0.5).cpu().numpy())
        ybt_targets.extend(y.cpu().numpy())

ybt_probs = np.array(ybt_probs)
ybt_predictions = np.array(ybt_predictions)
ybt_targets = np.array(ybt_targets)

print("\nYBT Test Set Performance:")
print(classification_report(ybt_targets, ybt_predictions, zero_division=0))
print(f"ROC-AUC: {roc_auc_score(ybt_targets, ybt_probs):.3f}")

# Threshold optimization for YBT
prec, rec, thresholds = precision_recall_curve(ybt_targets, ybt_probs)
f1s = 2 * (prec * rec) / (prec + rec + 1e-8)
best_thresh_idx = np.argmax(f1s)
best_threshold = thresholds[best_thresh_idx]

print(f"\nBest threshold for YBT: {best_threshold:.3f}")
ybt_predictions_optimal = (ybt_probs >= best_threshold).astype(int)
print(f"F1 at optimal threshold: {f1_score(ybt_targets, ybt_predictions_optimal):.3f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(ybt_probs, bins=50, alpha=0.7)
plt.title('YBT Prediction Probabilities Distribution')
plt.xlabel('Prediction Probability')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
plt.hist(ybt_probs[ybt_targets == 0], bins=30, alpha=0.7, label='Class 0', density=True)
plt.hist(ybt_probs[ybt_targets == 1], bins=30, alpha=0.7, label='Class 1', density=True)
plt.title('YBT Prediction Probabilities by Class')
plt.xlabel('Prediction Probability')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

# 9. comparison with prev models

In [ ]:
print("\n" + "="*60)
print("COMPREHENSIVE COMPARISON AND ANALYSIS")
print("="*60)

results_comparison = {
    'Model': ['Random Forest', 'Neural Network', 'TabNet', 'DANN (Original)', 'DANN (Improved)'],
    'F1_Score': [0.619, 0.667, 0.667, 0.667, f1_score(ybt_targets, ybt_predictions_optimal)],
    'ROC_AUC': [0.325, 0.523, 0.388, 0.500, roc_auc_score(ybt_targets, ybt_probs)],
    'Threshold': [0.159, 0.157, 0.116, 0.505, best_threshold]
}

comparison_df = pd.DataFrame(results_comparison)
print("\nPerformance Comparison:")
print(comparison_df)

# Save model
os.makedirs('/Users/eb2007/playground/bullpy/c4_play2/models', exist_ok=True)
torch.save(model.state_dict(), '/Users/eb2007/playground/bullpy/c4_play2/models/dann_improved.pth')

print("\nImproved DANN model saved successfully!")
print("Domain adaptation experiment completed!")

# Additional analysis
print(f"\nDetailed Analysis:")
print(f"- Model stopped at epoch: {trainer.current_epoch}")
print(f"- Best validation F1: {trainer.checkpoint_callback.best_model_score:.3f}")
print(f"- Final learning rate: {trainer.optimizers[0].param_groups[0]['lr']:.6f}")
print(f"- Prediction range: [{ybt_probs.min():.3f}, {ybt_probs.max():.3f}]")
print(f"- Mean prediction: {ybt_probs.mean():.3f}")
print(f"- Std prediction: {ybt_probs.std():.3f}")